# Optimization — ONNX Runtime FP32 and INT8 vs TorchScript baseline

**Hypothesis (written before testing):** for this small 1D-CNN served on CPU with batch 1, ONNX Runtime (graph optimisations, fused Conv+BN+ReLU) lowers mean and p95 latency versus TorchScript; static INT8 quantization further reduces model file size by ~4x, with a macro-F1 drop < 0.02 on the test split.

**Fair comparison conditions:** same bundle (same weights), same test split, same batch sizes (1, 8, 32), 1 intra-op thread, 20 warm-up + 300 timed calls, each runtime in its own process, same machine in the same session.

Run after `benchmark.ipynb` (same session: data + `artifacts/bundle` present).

In [ ]:
%cd /content/project
!python ai-infrastructure/optimization/quantize.py --bundle artifacts/bundle --mode static
!python ai-infrastructure/benchmarks/benchmark.py --bundle artifacts/bundle --runtimes torch onnx onnx-int8 --threads 1 --out ai-infrastructure/optimization/results

In [ ]:
import pandas as pd
df = pd.read_csv("ai-infrastructure/optimization/results/baseline-results.csv")
base = df[df.runtime == "torch"].set_index("batch_size")
rows = []
for rt in ["onnx", "onnx-int8"]:
    o = df[df.runtime == rt].set_index("batch_size")
    for bs in o.index:
        rows.append({"optimized": rt, "batch_size": bs,
                     "mean_ms base→opt": f"{base.loc[bs,'mean_ms']:.3f} → {o.loc[bs,'mean_ms']:.3f}",
                     "p95_ms base→opt": f"{base.loc[bs,'p95_ms']:.3f} → {o.loc[bs,'p95_ms']:.3f}",
                     "speedup (mean)": round(base.loc[bs,'mean_ms'] / o.loc[bs,'mean_ms'], 2),
                     "throughput base→opt": f"{base.loc[bs,'throughput_samples_s']:.0f} → {o.loc[bs,'throughput_samples_s']:.0f}",
                     "RSS MB base→opt": f"{base.loc[bs,'rss_after_load_mb']:.0f} → {o.loc[bs,'rss_after_load_mb']:.0f}",
                     "file MB base→opt": f"{base.loc[bs,'model_file_mb']:.3f} → {o.loc[bs,'model_file_mb']:.3f}",
                     "macro-F1 base→opt": f"{base.loc[bs,'test_macro_f1']:.3f} → {o.loc[bs,'test_macro_f1']:.3f}"})
comparison = pd.DataFrame(rows); comparison.to_csv("ai-infrastructure/optimization/results/comparison.csv", index=False)
comparison

In [ ]:
import matplotlib.pyplot as plt
raw = pd.read_csv("ai-infrastructure/optimization/results/raw-latencies.csv")
fig, ax = plt.subplots(figsize=(7, 4))
for rt, g in raw[raw.batch_size == 1].groupby("runtime"):
    ax.hist(g.latency_ms, bins=60, alpha=.6, label=rt)
ax.set_xlabel("latency (ms), batch 1, 1 thread"); ax.set_ylabel("calls"); ax.legend()
plt.savefig("ai-infrastructure/optimization/results/latency-hist.png", dpi=150, bbox_inches="tight")

## Interpretation (fill in from the table above — do not pre-write numbers)
- Latency / throughput: …
- Memory and file size: …
- Quality: …
- Verdict: keep TorchScript / switch the API to `MODEL_RUNTIME=onnx` / `onnx-int8`, and why. A slower optimized model is a valid result if explained (e.g. quantize/dequantize overhead dominates for a model of a few thousand parameters (MLP: 4,229)).